# MATH840 Final project — foundation and neural models (Colab)

The companion of `LUKIANETS_project_v2.ipynb`. Same data, same **20 rolling 48-hour windows** (one origin every midnight,
6–25 October), same scores, judged on day 2 (the private hours); at the end
it writes `colab_bt.csv` and `colab_final.csv`, which the Mac notebook merges (Section 2.10) into one scoreboard.

It runs in the Colab web page **or** in VS Code connected to a Colab runtime. A GPU runtime (T4 or better) is needed.

**Keys** — never paste a key into a cell:
- `NIXTLA_API_KEY` for TimeGPT;
- `TABPFN_TOKEN` for TabPFN: open https://ux.priorlabs.ai, log in, accept the licence on the *Licenses* tab, and copy the
  API key from https://ux.priorlabs.ai/account. The key alone is not enough: the licence of the TabPFN **model** must be
  accepted too, or TabPFN stops with "requires a one-time license acceptance". Without both, TabPFN-TS is skipped.

| | Colab in the browser | VS Code + Colab runtime |
|---|---|---|
| GPU | Runtime → Change runtime type → T4 GPU | choose a GPU runtime when you connect |
| Data files | the template cell asks you to upload them | right-click `train.csv` and `weather_forecast.csv` in the VS Code **Explorer → Upload to Colab** (or the upload widget below) |
| Keys | Colab **secrets** (key icon on the left) | Colab secrets do **not** work in VS Code: put both keys in the `.env` file (`NAME=value`, one per line) and **Upload to Colab** it too |
| Results | downloaded by the last cell | the last cell saves them to your Google Drive, folder `MATH840` (`files.download` does not work in VS Code) |

Then run the install cell; if pip asks to restart, restart the session and run from the top, skipping the install cell.
Run all: a model whose package failed to install is reported with ✗ and skipped; the rest still run.

⚠️ `.env` holds your keys: keep it out of the public GitHub repository (add `.env` to `.gitignore`).

In [15]:
# Core: the template's libraries and the course's foundation models
!pip install -q statsforecast==2.1.1 chronos-forecasting==2.3.2 "timesfm[torch]==3.0.2" tirex-ts==1.4.2 nixtla python-dotenv lightgbm
# 2026 models -- each line may fail on its own; the notebook then skips that model
!pip install -q tirex-2                                              # TiRex-2 (NX-AI): covariates
!pip install -q toto-models                                          # Toto 2.0 (Datadog)
!pip install -q "git+https://github.com/mistralai/timestral.git"     # Timestral (Mistral, Oct 2026): covariates
!pip install -q tabpfn                                               # TabPFN v2 (Prior Labs), for TabPFN-TS
# Neural networks trained on this series
!pip install -q neuralforecast

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [16]:
# The data files: already on the runtime (uploaded with "Upload to Colab"), or through the widget below.
from pathlib import Path
from IPython.display import display

NEEDED = ["train.csv", "weather_forecast.csv"]


def save_uploads(widget):
    # writes what the upload widget received to the runtime's disk (ipywidgets 7 and 8 store it differently)
    items = widget.value.values() if isinstance(widget.value, dict) else widget.value
    for item in items:
        name = item["metadata"]["name"] if "metadata" in item else item["name"]
        Path(name).write_bytes(bytes(item["content"]))
        print("saved", name)


if not all(Path(f).exists() for f in NEEDED):
    if "uploader" in globals() and uploader.value:
        save_uploads(uploader)
    else:
        try:
            import ipywidgets as widgets
            uploader = widgets.FileUpload(multiple=True, description="train.csv, weather_forecast.csv, .env")
            display(uploader)
        except ImportError:
            pass
        raise FileNotFoundError(f"{NEEDED} are not on the runtime. Upload them (VS Code: right-click -> Upload to Colab, "
                                "or choose them in the widget above), then run this cell again.")
print("on the runtime:", sorted(p.name for p in Path(".").iterdir() if p.is_file()))

on the runtime: ['.env', 'colab_bt.csv', 'colab_final.csv', 'train.csv', 'weather_forecast.csv']


In [17]:
# Keys: Colab secrets in the browser; in VS Code (no secrets there) the uploaded .env file. Only presence is printed.
import os
for key in ("NIXTLA_API_KEY", "TABPFN_TOKEN"):
    try:
        from google.colab import userdata
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
try:
    from dotenv import load_dotenv
    for f in (".env", "/content/.env"):
        if Path(f).exists():
            load_dotenv(f, override=False)
except ImportError:
    pass
for key in ("NIXTLA_API_KEY", "TABPFN_TOKEN"):
    print(f"{key}: {'found' if os.environ.get(key) else 'missing -- that model will be skipped'}")

NIXTLA_API_KEY: found
TABPFN_TOKEN: found


## 0. Setup

Download `train.csv` and `weather_forecast.csv` from Moodle. In Colab, the next cell asks you to upload
them. On your own machine, put them in the same folder as the notebook.

In [18]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

SURNAME = "LUKIANETS"   # <-- for your submission file name

if not SURNAME.strip():
    raise ValueError("Set SURNAME before running the rest.")

In [19]:
NEEDED = ["train.csv", "weather_forecast.csv"]

if not all(Path(f).exists() for f in NEEDED):
    try:
        from google.colab import files
        files.upload()                       # choose both files at once
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} next to this notebook.")

data = pd.read_csv("train.csv", parse_dates=["ds"])
weather_fc = pd.read_csv("weather_forecast.csv", parse_dates=["ds"])

H = 48                                       # hours to forecast
M = 24                                       # the benchmark's season: one day
FUTURE = weather_fc["ds"]

assert len(FUTURE) == H and FUTURE.iloc[0] == data["ds"].iloc[-1] + pd.Timedelta(hours=1)
print(f"history: {len(data)} hours, {data['ds'].min()} to {data['ds'].max()}")
print(f"you are forecasting: {FUTURE.iloc[0]} to {FUTURE.iloc[-1]}")
data.tail()

history: 33480 hours, 2023-01-01 01:00:00 to 2026-10-27 00:00:00
you are forecasting: 2026-10-27 01:00:00 to 2026-10-29 00:00:00


,ds,y,cloud_cover,temperature,wind_speed,pressure,humidity,daylight,air_density
33475,2026-10-26 20:00:00,42.5,1.0,9,3,763,87,0,1.259
33476,2026-10-26 21:00:00,76.7,1.0,9,4,763,87,0,1.259
33477,2026-10-26 22:00:00,65.8,1.0,7,3,763,100,0,1.268
33478,2026-10-26 23:00:00,69.9,1.0,7,3,763,100,0,1.268
33479,2026-10-27 00:00:00,69.9,1.0,7,3,763,100,0,1.268


### The data

| Column | Meaning |
|---|---|
| `ds` | the hour |
| `y` | electricity generated in that hour; this is what you forecast |
| `cloud_cover` | from 0 (clear) to 1 (overcast), in steps of 0.25 |
| `temperature` | °C |
| `wind_speed` | m/s |
| `pressure` | mm Hg |
| `humidity` | % |
| `daylight` | 1 while the sun is above the horizon |
| `air_density` | kg/m³ |

`train.csv` has the weather as it was. `weather_forecast.csv` has a forecast for the 48 hours you
predict, of the kind a weather service issues. Its errors grow from the first day to the second.

## 1. How you are scored

Your file gets three scores on the 48 hidden hours, and lower is better for each. RMSE measures the
point forecast in the units of `y`, and MAPE measures it in percent of the actual value. The Winkler
score measures the 95% interval: its width, plus 40 times ($2/\alpha$) the distance by which the actual
value falls outside it.

The benchmark is a seasonal naive forecast with a daily season. Each hour of the next two days gets the
value of the same hour on the last day you have. Its 95% interval is the standard one for seasonal naive,
$\hat y \pm 1.96\,\hat\sigma\sqrt{k+1}$, where $k$ is the number of full days into the horizon.

The project description on Moodle explains the points and the leaderboard.

In [20]:
def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def mape(actual, forecast) -> float:
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return float(np.mean(np.abs((actual - forecast) / actual)) * 100)


def winkler(actual, lower, upper, alpha: float = 0.05) -> float:
    actual, lower, upper = (np.asarray(a, float) for a in (actual, lower, upper))
    below = np.clip(lower - actual, 0, None)
    above = np.clip(actual - upper, 0, None)
    return float(np.mean((upper - lower) + 2 / alpha * (below + above)))


def score(actual, forecast, lower, upper) -> dict:
    inside = (np.asarray(actual) >= lower) & (np.asarray(actual) <= upper)
    return {"RMSE": rmse(actual, forecast), "MAPE": mape(actual, forecast),
            "Winkler": winkler(actual, lower, upper), "coverage": float(inside.mean())}


def snaive(history: np.ndarray, h: int = H, m: int = M):
    # The benchmark: repeat the last season, with the seasonal-naive 95% interval.
    history = np.asarray(history, float)
    point = np.array([history[-m + (i % m)] for i in range(h)])
    sigma = np.std(history[m:] - history[:-m], ddof=1)
    k = np.arange(h) // m
    half = 1.96 * sigma * np.sqrt(k + 1)
    return point, np.clip(point - half, 0, None), point + half

### The benchmark on a validation window

You cannot score a forecast on the hidden hours, so the cell below holds back the last 48 hours of the
history, forecasts them from the hours before, and scores the result. This is a single window. You can
decide how much more validation you need.

In [21]:
y = data["y"].to_numpy(float)
past, valid = y[:-H], y[-H:]

bench = snaive(past)
results = pd.DataFrame({"seasonal naive": score(valid, *bench)}).T
results

,RMSE,MAPE,Winkler,coverage
seasonal naive,120.404,51.088,568.464,1.0


## 2. Foundation and neural models

### 2.0 The same protocol as on the Mac

In [22]:
import itertools
import os
import time
import logging
os.environ["NIXTLA_ID_AS_COL"] = "true"
for name in ("transformers", "root", "cmdstanpy", "prophet", "lightning", "lightning.pytorch", "gluonts",
             "nixtla.nixtla_client", "pytorch_lightning", "neuralforecast"):
    logging.getLogger(name).setLevel(logging.ERROR)

from IPython.display import display
from scipy import signal, stats
from scipy.special import boxcox
from coreforecast.scalers import boxcox_lambda
from statsmodels.tsa.seasonal import MSTL as MSTL_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, AutoTheta, MSTL
try:
    from dotenv import load_dotenv
    load_dotenv()                  # .env next to the notebook holds NIXTLA_API_KEY: never paste the key here
except ImportError:
    pass

dates = data["ds"].reset_index(drop=True)
FREQ = "h"
DAY = 24
WINDOWS = 20                       # rolling 48-hour windows: the SAME on the Mac and in Colab (merged later)
STEP = 24                          # hours between window origins: one origin every midnight (6-25 October)
PART = "day2"                      # the hours a choice is judged on: "day2" (hours 25-48, the private part) or "all"
LAMBDA = 0.45                      # Box-Cox lambda (Guerrero on the last 12 weeks, Section 2.2)
FM_CONTEXT = 2048                  # hours of history a foundation model reads (default)

T0 = time.time()
def elapsed():
    print(f"[{(time.time() - T0) / 60:.1f} min since the start of Section 2]")

In [23]:
NAN = lambda h: np.full(h, np.nan)
FORECASTERS, MIXES, POST, FAMILY, FINAL = {}, {}, {}, {}, {}
bt = None
BASE_COLS = ("window", "step", "actual", "daylight", "benchmark", "benchmark lo", "benchmark hi")
PURGE = -(-H // STEP) - 1          # neighbours this close share hours with a window (1 for 48-hour windows a day apart)


def fwd(x, kind):
    x = np.asarray(x, float)
    if kind == "log":
        return np.log(x)
    if kind == "bc":
        return boxcox(x, LAMBDA)
    return x


def inv(z, kind):
    z = np.asarray(z, float)
    if kind == "log":
        return np.exp(z)
    if kind == "bc":
        return np.power(np.maximum(LAMBDA * z + 1, 1e-12), 1 / LAMBDA)
    return z


def weather_features(df: pd.DataFrame) -> pd.DataFrame:
    # the weather as regressors, plus the physical interactions trees and linear models cannot find alone
    df = df.reset_index(drop=True)
    out = df[["daylight", "cloud_cover", "temperature", "wind_speed", "humidity", "pressure", "air_density"]].astype(float).copy()
    out["sun"] = out["daylight"] * (1 - out["cloud_cover"])         # clear sky while the sun is up
    out["humid_day"] = out["daylight"] * out["humidity"]             # humidity matters while the sun is up
    out["wind3"] = out["wind_speed"] ** 3 / 100                      # wind power grows with the cube of the speed
    # Where the sun is, from the daylight column alone (no outside data): how long the day is, how far into its
    # daylight hours an hour lies, a clear-sky curve peaking at solar noon, and the season. 0/1 daylight cannot say
    # that late-October days are shorter and the sun lower, which is what the models missed in window 1.
    ds = pd.to_datetime(df["ds"])
    day = ds.dt.normalize().to_numpy()
    light = out["daylight"].to_numpy()
    length = pd.Series(light).groupby(day).transform("sum").to_numpy()
    order = pd.Series(light).groupby(day).cumsum().to_numpy()
    position = np.where(light == 1, (order - 0.5) / np.maximum(length, 1), 0.0)
    out["day_length"] = length
    out["sun_position"] = position
    out["clear_sky"] = np.where(light == 1, np.sin(np.pi * position), 0.0)
    out["doy_sin"] = np.sin(2 * np.pi * ds.dt.dayofyear.to_numpy() / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * ds.dt.dayofyear.to_numpy() / 365.25)
    return out


FEAT_DF = weather_features(pd.concat([data.drop(columns="y"), weather_fc], ignore_index=True))   # history, then the forecast
FEAT_COLS = list(FEAT_DF.columns)
FEAT = FEAT_DF.to_numpy()
ALL_DATES = pd.concat([dates, FUTURE], ignore_index=True)
COV_COLS = ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3"]   # the weather
SUN_COLS = ["day_length", "sun_position", "clear_sky", "doy_sin", "doy_cos"]                           # the sun


def cov(cols, start, stop):
    # covariate rows (len(cols), stop - start) for the absolute hours start .. stop-1 (the forecast after the history)
    return FEAT_DF[cols].to_numpy(float)[start:stop].T


def _clean(out, h) -> dict:
    get = lambda *keys: next((np.asarray(out[k], float).ravel()[:h] for k in keys if k in out), NAN(h))
    return {"mean": get("mean"), "lo": get("lo", "lo-95"), "hi": get("hi", "hi-95")}


def on_scale(fit_predict, transform="log"):
    # fit_predict(x, end, h) sees fwd(y[:end]); its forecasts come back on the original scale
    def forecast(end, h):
        out = _clean(fit_predict(fwd(y[:end], transform), end, h), h)
        out = {k: inv(v, transform) for k, v in out.items()}
        out["mean"], out["lo"] = np.maximum(out["mean"], 0), np.maximum(out["lo"], 0)
        return out
    return forecast


def sf(make, transform="log", context=1344, level=True):
    # a statsforecast model: make() builds a fresh one, fitted on the last `context` hours
    def fit_predict(x, end, h):
        x = x[-context:] if context else x
        return make().forecast(y=x, h=h, level=[95]) if level else make().forecast(y=x, h=h)
    return on_scale(fit_predict, transform)


FORECASTERS["benchmark"] = lambda end, h: dict(zip(("mean", "lo", "hi"), snaive(y[:end], h)))


def window_ends(windows=None):
    # where each window ends: window 1 ends with the history, the others STEP hours apart before it
    return [len(y) - (k - 1) * STEP for k in range(windows or WINDOWS, 0, -1)]


def backtest(forecasters: dict, windows=None) -> pd.DataFrame:
    # every forecaster on every window, each window forecast from the hours before it only
    windows = windows or WINDOWS
    ends = window_ends(windows)
    frames = []
    for k, e in zip(range(windows, 0, -1), ends):
        p, lo, hi = snaive(y[:e - H])
        frames.append(pd.DataFrame({"window": k, "step": np.arange(1, H + 1), "actual": y[e - H:e],
                                    "daylight": data["daylight"].to_numpy()[e - H:e],
                                    "benchmark": p, "benchmark lo": lo, "benchmark hi": hi}))
    frame = pd.concat(frames, ignore_index=True)
    bench_rmse = np.mean([rmse(g["actual"], g["benchmark"]) for _, g in frame.groupby("window")])
    seconds = {}
    for name, f in forecasters.items():
        start = time.time()
        try:
            parts = [_clean(f(e - H, H), H) for e in ends]
            if any(np.isnan(p["mean"]).any() for p in parts):
                raise ValueError("NaN in the forecast")
        except Exception as err:
            print(f"  ✗ {name}: {type(err).__name__}: {str(err)[:200]}")
            continue
        FORECASTERS[name] = f
        for key, col in (("mean", name), ("lo", f"{name} lo"), ("hi", f"{name} hi")):
            frame[col] = np.concatenate([p[key] for p in parts])
        seconds[name] = time.time() - start
        r = np.mean([rmse(g["actual"], g[name]) for _, g in frame.groupby("window")])
        print(f"  ✓ {name:<46s} RMSE {r:8.1f}  (skill {r / bench_rmse:.3f})   {seconds[name]:.1f}s", flush=True)
    frame.attrs["seconds"] = seconds
    return frame


def add(bt, new):
    out = bt.drop(columns=[c for c in new.columns if c in bt.columns and c not in BASE_COLS])
    out = pd.concat([out, new[[c for c in new.columns if c not in out.columns]]], axis=1)
    out.attrs["seconds"] = {**bt.attrs.get("seconds", {}), **new.attrs.get("seconds", {})}
    return out


def forecasts_in(bt) -> list[str]:
    return [c for c in bt.columns if c not in BASE_COLS and not c.endswith((" lo", " hi"))] + ["benchmark"]


def _part(frame, part=None):
    # the rows of the judged hours: day 2 = steps 25-48 (the private part), day 1 = steps 1-24, all = 1-48
    part = part or PART
    if part == "day2":
        return frame[frame["step"] > DAY]
    if part == "day1":
        return frame[frame["step"] <= DAY]
    return frame


def _per_window(rows, f, lo=None, hi=None):
    # RMSE, MAPE (and Winkler, coverage) of forecast f in each window of `rows`, as arrays (fast: no groupby)
    _, ix = np.unique(rows["window"].to_numpy(), return_inverse=True)
    n = np.bincount(ix)
    a, f = rows["actual"].to_numpy(float), np.asarray(f, float)
    out = [np.sqrt(np.bincount(ix, (a - f) ** 2) / n), np.bincount(ix, np.abs((a - f) / a)) / n * 100]
    if lo is not None:
        lo, hi = np.asarray(lo, float), np.asarray(hi, float)
        miss = np.clip(lo - a, 0, None) + np.clip(a - hi, 0, None)
        out += [np.bincount(ix, (hi - lo) + 40 * miss) / n, np.bincount(ix, ((a >= lo) & (a <= hi)).astype(float)) / n]
    return out


def window_table(columns=None, per_window=False, part=None) -> pd.DataFrame:
    # mean over the windows of the template's scores on the judged hours; skill = score / benchmark's;
    # won = windows with the lowest RMSE; beat bench = windows with a lower RMSE than the seasonal naive
    columns = list(dict.fromkeys((forecasts_in(bt) if columns is None else list(columns)) + ["benchmark"]))
    rows = _part(bt, part)
    ids = np.unique(rows["window"].to_numpy())
    stats_, per = {}, {}
    for c in columns:
        has = rows[f"{c} lo"].notna().all()
        r, m, *wc = _per_window(rows, rows[c], rows[f"{c} lo"] if has else None, rows[f"{c} hi"] if has else None)
        stats_[c] = {"RMSE": r.mean(), "MAPE": m.mean(), "Winkler": wc[0].mean() if has else np.nan,
                     "coverage": wc[1].mean() if has else np.nan}
        per[c] = pd.Series(r, index=ids)
    tab = pd.DataFrame(stats_).T
    b = tab.loc["benchmark"]
    tab["RMSE skill"], tab["MAPE skill"] = tab["RMSE"] / b["RMSE"], tab["MAPE"] / b["MAPE"]
    tab["point skill"] = (tab["RMSE skill"] + tab["MAPE skill"]) / 2
    tab["Winkler skill"] = tab["Winkler"] / b["Winkler"]
    w_rmse = pd.DataFrame(per)
    tab["won"] = [int((w_rmse.idxmin(axis=1) == c).sum()) for c in tab.index]
    tab["beat bench"] = [int((w_rmse[c] < w_rmse["benchmark"]).sum()) for c in tab.index]
    tab["seconds"] = [bt.attrs.get("seconds", {}).get(c, np.nan) for c in tab.index]
    tab = tab.sort_values("point skill")
    if per_window:
        return tab, (w_rmse.T / w_rmse["benchmark"]).loc[tab.index].round(2)
    return tab


def run(family: str, forecasters: dict, rerun=False) -> pd.DataFrame:
    global bt
    todo = {n: f for n, f in forecasters.items() if rerun or bt is None or n not in bt.columns}
    if todo:
        print(f"{family}: {len(todo)} forecaster(s) x {WINDOWS} windows")
        new = backtest(todo)
        bt = new if bt is None else add(bt, new)
    for n in forecasters:
        if n in bt.columns:
            FAMILY[n] = family
            FINAL.pop(n, None)
    return window_table([n for n in forecasters if n in bt.columns])


def best_of(family: str) -> str:
    names = [n for n, f in FAMILY.items() if f == family and n in bt.columns]
    return window_table(names).drop(index="benchmark")["point skill"].idxmin()


def combine(names, label, how="mean"):
    names = [n for n in dict.fromkeys(names) if n in bt.columns]
    MIXES[label], FAMILY[label] = (names, how), "combination"
    FINAL.pop(label, None)
    agg = (lambda cols, **kw: bt[cols].median(axis=1, **kw)) if how == "median" else (lambda cols, **kw: bt[cols].mean(axis=1, **kw))
    bt[label] = agg(names)
    bt[f"{label} lo"] = agg([f"{n} lo" for n in names], skipna=False)
    bt[f"{label} hi"] = agg([f"{n} hi" for n in names], skipna=False)


def others(w):
    # the windows that share no hour with window w: w and its PURGE neighbours on each side are left out
    return bt[(bt["window"] - w).abs() > PURGE]


def _radius(actual, point, daylight, recipe):
    # the 95% conformal radius from past errors: additive, or additive split day / night
    err = np.abs(np.asarray(actual, float) - np.asarray(point, float))
    if recipe == "conformal day/night":
        return {d: np.quantile(err[np.asarray(daylight) == d], 0.95) for d in (0, 1)}
    return np.quantile(err, 0.95)


def _bounds(point, daylight, r, recipe):
    point = np.asarray(point, float)
    rr = np.where(np.asarray(daylight) == 1, r[1], r[0]) if recipe == "conformal day/night" else r
    return np.maximum(point - rr, 0), point + rr


SCALES = np.round(np.arange(0.5, 1.501, 0.05), 2)


def _scaled(point, lo, hi, c):
    # the interval stretched (c > 1) or shrunk (c < 1) around the point forecast
    point, lo, hi = (np.asarray(v, float) for v in (point, lo, hi))
    return np.maximum(point - c * (point - lo), 0), point + c * (hi - point)


def _best_scale(rows, label):
    # the width factor with the lowest mean Winkler on `rows`
    p, lo, hi = rows[label], rows[f"{label} lo"], rows[f"{label} hi"]
    return min(SCALES, key=lambda c: _per_window(rows, p, *_scaled(p, lo, hi, c))[2].mean())


RECIPES = ["conformal", "conformal day/night"]


def interval_check(label: str, part=None) -> pd.DataFrame:
    # every recipe judged on windows it did not see: radius / width factor from the windows sharing no hour with it
    rows = []
    for w in sorted(bt["window"].unique()):
        g, o = _part(bt[bt["window"] == w], part), _part(others(w), part)
        recipes = {rec: _bounds(g[label], g["daylight"], _radius(o["actual"], o[label], o["daylight"], rec), rec) for rec in RECIPES}
        if g[f"{label} lo"].notna().all():
            recipes["own interval"] = (g[f"{label} lo"].to_numpy(), g[f"{label} hi"].to_numpy())
            recipes["own interval x c"] = _scaled(g[label], g[f"{label} lo"], g[f"{label} hi"], _best_scale(o, label))
        for rec, (lo, hi) in recipes.items():
            s = score(g["actual"], g[label], lo, hi)
            rows.append({"recipe": rec, "window": w, "Winkler": s["Winkler"], "coverage": s["coverage"],
                         "width": float(np.mean(np.asarray(hi) - np.asarray(lo)))})
    return pd.DataFrame(rows).groupby("recipe")[["Winkler", "coverage", "width"]].mean().sort_values("Winkler")


SHADE_FACTORS = np.round(np.arange(0.1, 1.001, 0.05), 2)
SHADE_LEVELS = (50, 100, 150, 200, 300)


def _shade(point, a, level):
    # the night rule: forecasts below `level` multiplied by a (MAPE punishes over-forecasting small values hardest)
    point = np.asarray(point, float)
    return np.where(point < level, a * point, point)


def _tune_post(frame, base, factors=SHADE_FACTORS):
    # the night rule (a, level) with the lowest RMSE skill + MAPE skill on the judged hours of `frame`
    # (base: a Series on frame's index); returns a, level and the RMSE skill + MAPE skill reached
    rows = _part(frame)
    b = base[rows.index].to_numpy()
    br, bm = _per_window(rows, rows["benchmark"])
    obj = lambda a, level: sum(v.mean() / ref.mean() for v, ref in zip(_per_window(rows, _shade(b, a, level)), (br, bm)))
    a, level = min(((a, L) for a in factors for L in SHADE_LEVELS), key=lambda p: obj(*p))
    return float(a), int(level), float(obj(a, level))


def postprocess(day_label, night_label=None, name=None, shade=True) -> str:
    # 1) the day model in daylight hours; at night the same model or its log-scale version, whichever scores
    #    better (night_label fixes it);
    # 2) the night rule: forecasts below `level` multiplied by a (shade=False: no rule).
    # The interval: the one of the model used at each hour, stretched to include the point; its width is judged in
    # interval_check like any model's. The night model, a and level are tuned on the windows sharing no hour with
    # the scored one (purged leave-one-window-out), so bt holds an honest score; the submission uses them tuned on
    # all windows.
    log = f"{day_label} log"
    nights = [night_label] if night_label else [day_label] + ([log] if log in bt.columns else [])
    name = name or f"{day_label} [{'post' if shade else 'hybrid'}]"
    factors = SHADE_FACTORS if shade else (1.0,)
    night = bt["daylight"].to_numpy() == 0
    pick = lambda n, s: pd.Series(np.where(night, bt[n + s], bt[day_label + s]), index=bt.index)

    def tune(frame):
        # [night model, a, level]: the night model whose tuned rule reaches the lowest RMSE skill + MAPE skill
        fits = {n: _tune_post(frame, pick(n, "")[frame.index], factors) for n in nights}
        n = min(fits, key=lambda k: fits[k][2])
        return [n, *fits[n][:2]]

    point, plo, phi = (pd.Series(np.nan, index=bt.index) for _ in range(3))
    chosen = []
    for w in sorted(bt["window"].unique()):
        n, a, level = tune(others(w))
        chosen.append(("log" if n == log else "same" if n == day_label else n, a, level))
        idx = bt.index[bt["window"] == w]
        point[idx] = _shade(pick(n, "")[idx], a, level)
        plo[idx] = np.minimum(pick(n, " lo")[idx], point[idx])
        phi[idx] = np.maximum(pick(n, " hi")[idx], point[idx])
    bt[name], bt[f"{name} lo"], bt[f"{name} hi"] = point, plo, phi
    n, a, level = tune(bt)
    POST[name], FAMILY[name] = {"day": day_label, "night": n, "a": a, "level": level}, "post-processed"
    FINAL.pop(name, None)
    print(f"{name}: night = {n}; submission rule a = {a}, level = {level}; held-out windows chose (night, a, level) "
          f"{pd.Series([str(x) for x in chosen]).value_counts().to_dict()}")
    return name


def from_all_of_y(label: str) -> dict:
    # every forecaster inside `label` once more on the WHOLE history (cached), averaged / post-processed as in bt
    if label not in FINAL:
        if label in MIXES:
            names, how = MIXES[label]
            parts = [from_all_of_y(n) for n in names]
            agg = np.median if how == "median" else np.mean
            FINAL[label] = {k: agg([p[k] for p in parts], axis=0) for k in ("mean", "lo", "hi")}
        elif label in POST:
            p = POST[label]
            d_, n_ = from_all_of_y(p["day"]), from_all_of_y(p["night"])
            night = weather_fc["daylight"].to_numpy()[:H] == 0
            base, lo, hi = (np.where(night, n_[k], d_[k]) for k in ("mean", "lo", "hi"))
            point = _shade(base, p["a"], p["level"])
            FINAL[label] = {"mean": point, "lo": np.minimum(lo, point), "hi": np.maximum(hi, point)}
        else:
            FINAL[label] = _clean(FORECASTERS[label](len(y), H), H)
    return FINAL[label]


def final_forecast(label: str, interval: str):
    # the 48 hours from the whole history; a conformal radius or width factor is fitted for each day of the horizon
    # separately (on all windows' day-1 errors for hours 1-24, on their day-2 errors for hours 25-48)
    fc = from_all_of_y(label)
    point = np.maximum(fc["mean"], 0)
    lo, hi = np.array(fc["lo"], float), np.array(fc["hi"], float)
    light = weather_fc["daylight"].to_numpy()[:H]
    for part, sl in (("day1", slice(0, DAY)), ("day2", slice(DAY, H))):
        rows = _part(bt, part)
        if interval == "own interval x c":
            lo[sl], hi[sl] = _scaled(point[sl], fc["lo"][sl], fc["hi"][sl], _best_scale(rows, label))
        elif interval in RECIPES:
            r = _radius(rows["actual"], rows[label], rows["daylight"], interval)
            lo[sl], hi[sl] = _bounds(point[sl], light[sl], r, interval)
        elif interval != "own interval":
            raise ValueError("interval: 'own interval', 'own interval x c' or one of RECIPES")
    if np.isnan(lo).any():
        raise ValueError(f"{label} has no interval of its own: use a conformal recipe")
    return point, np.minimum(np.maximum(lo, 0), point), np.maximum(hi, point)


def plot_validation(names, window=1, show=72):
    g = bt[bt["window"] == window]
    end = len(y) - (window - 1) * STEP
    x = dates.iloc[end - H:end]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(dates.iloc[end - H - show:end - H], y[end - H - show:end - H], color="grey", lw=0.8, label="history")
    ax.plot(x, g["actual"], color="black", lw=1.6, label="actual")
    ax.plot(x, g["benchmark"], color="grey", ls="--", lw=1.2, label="seasonal naive")
    ax.axvline(x.iloc[DAY - 1], color="grey", ls=":", lw=1)
    for n in names:
        ax.plot(x, g[n], lw=1.4, label=f"{n} (RMSE {rmse(g['actual'], g[n]):.1f})")
        if len(names) == 1 and g[f"{n} lo"].notna().all():
            ax.fill_between(x, g[f"{n} lo"], g[f"{n} hi"], alpha=0.2, label="95% interval")
    ax.set_title(f"window {window}" + (" (the template's validation window)" if window == 1 else "") + ": day 1 | day 2")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


RESULTS_DIR = "results"   # the exported model tables (<prefix>_bt.csv, <prefix>_final.csv) live here


def export(prefix: str):
    # the rolling-window table and each model's 48-hour forecast from the whole history, for merging elsewhere
    names = [n for n in forecasts_in(bt) if n != "benchmark" and n not in MIXES and n not in POST]
    Path(RESULTS_DIR).mkdir(exist_ok=True)
    bt[list(BASE_COLS) + [c for n in names for c in (n, f"{n} lo", f"{n} hi")]].to_csv(f"{RESULTS_DIR}/{prefix}_bt.csv", index=False)
    rows = []
    for n in names:
        try:
            fc = from_all_of_y(n)
        except Exception as err:
            print(f"  ✗ final forecast of {n}: {type(err).__name__}: {str(err)[:120]}")
            continue
        rows += [{"model": n, "family": FAMILY.get(n, "?"), "step": i + 1, "mean": fc["mean"][i], "lo": fc["lo"][i],
                  "hi": fc["hi"][i]} for i in range(H)]
    pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/{prefix}_final.csv", index=False)
    print(f"saved {RESULTS_DIR}/{prefix}_bt.csv ({len(names)} models) and {RESULTS_DIR}/{prefix}_final.csv")


def merge(prefix: str):
    # adds another notebook's exported models to bt, FAMILY and FINAL (the same windows are checked)
    global bt
    other = pd.read_csv(f"{RESULTS_DIR}/{prefix}_bt.csv")
    assert len(other) == len(bt) and (other["window"].to_numpy() == bt["window"].to_numpy()).all() \
        and np.allclose(other["actual"], bt["actual"]), "not the same windows: WINDOWS and STEP must match"
    new = [c for c in other.columns if c not in BASE_COLS]
    seconds = bt.attrs.get("seconds", {})
    bt = pd.concat([bt.drop(columns=[c for c in new if c in bt.columns]), other[new]], axis=1)
    bt.attrs["seconds"] = seconds
    fin = pd.read_csv(f"{RESULTS_DIR}/{prefix}_final.csv")
    for n, g in fin.groupby("model"):
        g = g.sort_values("step")
        FINAL[n] = {k: g[k].to_numpy(float) for k in ("mean", "lo", "hi")}
        FAMILY[n] = g["family"].iloc[0]
    print(f"merged {fin['model'].nunique()} models from {prefix}")

### 2.1 Foundation models, zero-shot

Each model reads up to `FM_CONTEXT` hours. "+ weather" models also read the weather of the history and of the
horizon (actual weather on the windows, `weather_fc` in the submission) as covariates. A 95% interval of their own
where the model outputs the 2.5% / 97.5% quantiles (Chronos-2, Timestral, TimeGPT); otherwise the interval is
conformal (their quantiles stop at 10% / 90%).

**Models are loaded one at a time:** each is scored on all its variants, its 48-hour forecast is computed, and it is
deleted from the GPU before the next one loads (the GPU memory left in use is printed). A model that fails — a
missing package, out of memory — is skipped and the next one starts with a clean GPU. If an earlier run filled the
GPU, do **Runtime → Restart session** first.

| Model | Released by | Covariates here |
|---|---|---|
| Chronos-2 | Amazon, 2025 | yes |
| Chronos-Bolt | Amazon, 2024 | no |
| TimesFM 2.5 (+ XReg) | Google, 2025 | yes (in-context linear regression) |
| TimesFM 3.0 | Google, 2026 (non-commercial licence) | yes |
| TiRex, TiRex-2 | NX-AI, 2025 / 2026 | TiRex-2: yes |
| Toto 2.0 | Datadog, 2026 | no (not yet in 2.0) |
| Timestral 28M / 209M | Mistral, October 2026 | yes |
| TabPFN-TS | Prior Labs | yes (as table features) |
| TimeGPT | Nixtla (API: the series is sent to Nixtla) | yes |

**This 20-window run** keeps the models that read the weather (and Timestral and Chronos-2 without it, for reference)
and tries each on a grid: context length × covariates (weather / weather + sun) × scale (raw / log). Chronos-Bolt,
TimesFM 2.5, TiRex, Toto 2.0 and Timestral 28M — weaker without covariates — are in the 10-window run of 4 November.

⚠️ Pretraining corpora are public archives: a model may have seen data like this series. A zero-shot score far
better than every other model on every window would be the sign.

In [24]:
import gc
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
_models = {}


def loaded(name):
    if name in _models:
        return _models[name]
    if name in ("Chronos-2", "Chronos-Bolt"):
        from chronos import BaseChronosPipeline
        repo = "amazon/chronos-2" if name == "Chronos-2" else "amazon/chronos-bolt-base"
        model = BaseChronosPipeline.from_pretrained(repo, device_map=DEVICE)
    elif name == "TimesFM 2.5":
        import timesfm
        model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
        model.compile(timesfm.ForecastConfig(max_context=FM_CONTEXT, max_horizon=128, normalize_inputs=True,
                                             use_continuous_quantile_head=True, fix_quantile_crossing=True,
                                             return_backcast=True))       # needed by XReg; the horizon is then the LAST rows
    elif name == "TimesFM 3.0":
        from timesfm3 import TimesFM3Forecaster
        model = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch", device=DEVICE)
    elif name == "TiRex":
        import tirex
        model = tirex.load_model("NX-AI/TiRex", device=DEVICE)
    elif name == "TiRex-2":
        from tirex2 import load_model
        model = load_model("NX-AI/TiRex-2", device=DEVICE)
    elif name.startswith("Toto 2.0"):
        from toto2 import Toto2Model
        model = Toto2Model.from_pretrained(f"Datadog/Toto-2.0-{name.split()[-1]}").to(DEVICE).eval()
    elif name.startswith("Timestral"):
        from timestral import TimestralForecaster
        model = TimestralForecaster.from_pretrained(f"mistralai/Timestral-{name.split()[-1]}", device=DEVICE)
    elif name == "TimeGPT":
        from nixtla import NixtlaClient
        model = NixtlaClient()                                   # reads NIXTLA_API_KEY
    else:
        raise ValueError(name)
    _models[name] = model
    return model


def unload(name):
    # deletes a model and returns its GPU memory before the next model is loaded
    _models.pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"  freed {name}: {torch.cuda.memory_allocated() / 1e9:.2f} GB of GPU memory still in use")


def run_one_by_one(groups: dict, family="foundation"):
    # groups = {model to load: {forecaster name: forecaster}}. One model at a time: load it, score every variant on
    # the windows, compute its 48-hour forecast from the whole history while it is still loaded, then free it.
    for model_name, forecasters in groups.items():
        print(f"\n=== {model_name} ===")
        try:
            with torch.no_grad():                           # inference only: no activations kept for gradients
                display(run(family, forecasters))
                for n in forecasters:
                    if n in bt.columns:
                        try:
                            from_all_of_y(n)                # cached in FINAL: export() will not reload the model
                        except Exception as err:
                            print(f"  ✗ final forecast of {n}: {type(err).__name__}: {str(err)[:160]}")
        finally:
            unload(model_name)
            elapsed()


def window_of(x, end, context=None):
    # the context the model reads: the last `context` (default FM_CONTEXT) hours before `end`; x = transformed y[:end]
    start = max(0, end - (context or FM_CONTEXT))
    return np.asarray(x[start:], np.float32), start


def chronos2(cols=None, context=None):
    # cols: the covariates (None = the series alone); context: the hours it reads (None = FM_CONTEXT)
    def fit_predict(x, end, h):
        ctx, start = window_of(x, end, context)
        entry = {"target": ctx}
        if cols:
            entry["past_covariates"] = {c: cov([c], start, end)[0].astype(np.float32) for c in cols}
            entry["future_covariates"] = {c: cov([c], end, end + h)[0].astype(np.float32) for c in cols}
        q, _ = loaded("Chronos-2").predict_quantiles([entry], prediction_length=h, quantile_levels=[0.025, 0.5, 0.975])
        q = np.asarray(q[0]).reshape(-1, h, 3)[0]
        return {"mean": q[:, 1], "lo": q[:, 0], "hi": q[:, 2]}
    return fit_predict


def chronos_bolt(x, end, h):
    ctx, _ = window_of(x, end)
    q, _ = loaded("Chronos-Bolt").predict_quantiles(torch.tensor(ctx), prediction_length=h, quantile_levels=[0.5])
    return {"mean": np.asarray(q)[0, :, 0]}


def timesfm25(weather=False):
    def fit_predict(x, end, h):
        ctx, start = window_of(x, end)
        model = loaded("TimesFM 2.5")
        if not weather:
            _, q = model.forecast(horizon=h, inputs=[ctx])
            return {"mean": np.asarray(q)[0, -h:, 5]}            # with return_backcast the horizon is the last h rows
        point, _ = model.forecast_with_covariates(
            inputs=[ctx], dynamic_numerical_covariates={c: [cov([c], start, end + h)[0]] for c in COV_COLS},
            xreg_mode="xreg + timesfm")
        return {"mean": np.asarray(point[0])[:h]}
    return fit_predict


def timesfm3(cols=None, context=None):
    def fit_predict(x, end, h):
        ctx, start = window_of(x, end, context)
        kw = {"past_future_covariates": cov(cols, start, end + h).astype(np.float32)} if cols else {}
        out = loaded("TimesFM 3.0").predict(ctx, horizon=h, return_quantiles=True, **kw)
        return {"mean": np.asarray(out.forecast).ravel()[:h]}
    return fit_predict


def tirex1(x, end, h):
    ctx, _ = window_of(x, end)
    q, _ = loaded("TiRex").forecast(context=[torch.tensor(ctx)], prediction_length=h)
    return {"mean": np.asarray(q)[0, :, 4]}


def tirex2(cols=None, context=None):
    def fit_predict(x, end, h):
        from tirex2 import TimeseriesType
        ctx, start = window_of(x, end, context)
        future = torch.tensor(cov(cols, start, end + h), dtype=torch.float32) if cols else None
        ts = TimeseriesType(target=torch.tensor(ctx).unsqueeze(0), past_covariates=None, future_covariates=future)
        q = loaded("TiRex-2").forecast([ts], prediction_length=h, output_type="numpy")[0]   # (1, 9 quantiles, h)
        return {"mean": np.asarray(q)[0, 4, :h]}
    return fit_predict


def toto2(size="22m"):
    def fit_predict(x, end, h):
        ctx, _ = window_of(x, end)
        target = torch.tensor(ctx, device=DEVICE).reshape(1, 1, -1)
        with torch.no_grad():
            q = loaded(f"Toto 2.0 {size}").forecast(
                {"target": target, "target_mask": torch.ones_like(target, dtype=torch.bool),
                 "series_ids": torch.zeros(1, 1, dtype=torch.long, device=DEVICE)},
                horizon=h, decode_block_size=768, has_missing_values=False)       # (9 quantiles, 1, 1, h)
        return {"mean": q[4, 0, 0, :h].float().cpu().numpy()}
    return fit_predict


def timestral(size="209M", cols=None, context=None):
    def fit_predict(x, end, h):
        ctx, start = window_of(x, end, context)
        kw = {"known_covariates": cov(cols, start, end + h)[None].astype(np.float32)} if cols else {}
        fc = loaded(f"Timestral {size}").forecast(ctx[None, None, :], prediction_length=h, **kw)
        to_np = lambda t: t.detach().float().cpu().numpy() if hasattr(t, "detach") else np.asarray(t, float)
        q = to_np(fc.quantiles)[0, 0]                                   # (h, 21); on the GPU it is a CUDA tensor
        levels = to_np(fc.quantile_levels)
        at = lambda p: np.array([np.interp(p, levels, q[i]) for i in range(h)])
        return {"mean": at(0.5), "lo": at(0.025), "hi": at(0.975)}
    return fit_predict


def timegpt(cols=None, finetune_steps=0, context=None):
    def fit_predict(x, end, h):
        _, start = window_of(x, end, context)
        df = pd.DataFrame({"unique_id": "series", "ds": dates.iloc[start:end].to_numpy(), "y": x[start:]})
        X_df = None
        if cols:
            for c in cols:
                df[c] = FEAT_DF[c].to_numpy()[start:end]
            X_df = pd.DataFrame({"unique_id": "series", "ds": ALL_DATES.iloc[end:end + h].to_numpy(),
                                 **{c: FEAT_DF[c].to_numpy()[end:end + h] for c in cols}})
        fc = loaded("TimeGPT").forecast(df=df, X_df=X_df, h=h, freq=FREQ, level=[95], finetune_steps=finetune_steps)
        return {"mean": fc["TimeGPT"].to_numpy(), "lo": fc["TimeGPT-lo-95"].to_numpy(), "hi": fc["TimeGPT-hi-95"].to_numpy()}
    return fit_predict

device: cuda


In [25]:
# One model at a time: {model to load: its variants}. Comment out a block to skip that model.
# The grid: context length (hours read) x covariates (the weather / the weather + the sun) x scale (raw / log).
# Short contexts follow today's level; long ones (a year or more) let a model see last October -> November.
# Within a model the longest contexts run last, so if one runs out of GPU memory only it is lost.
COVS = {"weather": COV_COLS, "weather + sun": COV_COLS + SUN_COLS}


def variants(model, make, contexts, scales=(None,)):
    # names keep the 10-window run's names at the default context: "Timestral 209M + weather", "... + weather log"
    out = {}
    for c in contexts:
        for cov_name, cols in COVS.items():
            for scale in scales:
                name = f"{model} + {cov_name}" + ("" if c == FM_CONTEXT else f", ctx {c}") + (f" {scale}" if scale else "")
                out[name] = on_scale(make(cols, c), scale)
    return out


FOUNDATION = {
    "Timestral 209M": {"Timestral 209M": on_scale(timestral("209M"), None),            # without covariates, for reference
                       **variants("Timestral 209M", lambda cols, c: timestral("209M", cols, c),
                                  [336, 672, 2048, 4380, 8760, 17520, 26280, 32768], scales=(None, "log"))},
    # 26280 = 3 years; 32768 = Timestral's maximum, the whole history but its first 30 days
    "Timestral 123M": variants("Timestral 123M", lambda cols, c: timestral("123M", cols, c), [2048, 8760, 17520, 32768]),
    "Chronos-2":      {"Chronos-2": on_scale(chronos2(), None),
                       **variants("Chronos-2", lambda cols, c: chronos2(cols, c), [336, 672, 2048, 4380, 8192],
                                  scales=(None, "log"))},                                # 8192: Chronos-2's maximum
    "TimesFM 3.0":    variants("TimesFM 3.0", lambda cols, c: timesfm3(cols, c), [1024, 2048, 4380]),
    "TiRex-2":        variants("TiRex-2", lambda cols, c: tirex2(cols, c), [2048]),
}
if os.environ.get("NIXTLA_API_KEY"):                             # an API: nothing on the GPU, but every call is billed
    FOUNDATION["TimeGPT"] = variants("TimeGPT", lambda cols, c: timegpt(cols, context=c), [2048, 8760])
print({m: len(v) for m, v in FOUNDATION.items()}, "variants")
run_one_by_one(FOUNDATION)
display(window_table([n for n, f in FAMILY.items() if f == "foundation"]))

{'Timestral 209M': 33, 'Timestral 123M': 8, 'Chronos-2': 21, 'TimesFM 3.0': 6, 'TiRex-2': 2, 'TimeGPT': 4} variants

=== Timestral 209M ===
foundation: 33 forecaster(s) x 20 windows


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

  ✓ Timestral 209M                                 RMSE    263.8  (skill 0.843)   3.4s
  ✓ Timestral 209M + weather, ctx 336              RMSE    232.0  (skill 0.742)   1.7s
  ✓ Timestral 209M + weather, ctx 336 log          RMSE    234.4  (skill 0.749)   1.7s
  ✓ Timestral 209M + weather + sun, ctx 336        RMSE    231.3  (skill 0.740)   1.8s
  ✓ Timestral 209M + weather + sun, ctx 336 log    RMSE    234.3  (skill 0.749)   1.8s
  ✓ Timestral 209M + weather, ctx 672              RMSE    214.6  (skill 0.686)   1.7s
  ✓ Timestral 209M + weather, ctx 672 log          RMSE    219.2  (skill 0.701)   1.7s
  ✓ Timestral 209M + weather + sun, ctx 672        RMSE    216.0  (skill 0.691)   2.0s
  ✓ Timestral 209M + weather + sun, ctx 672 log    RMSE    223.1  (skill 0.713)   2.0s
  ✓ Timestral 209M + weather                       RMSE    200.7  (skill 0.642)   3.2s
  ✓ Timestral 209M + weather log                   RMSE    208.9  (skill 0.668)   3.2s
  ✓ Timestral 209M + weather + sun         

,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"Timestral 209M + weather, ctx 32768",206.756,135.137,978.539,0.954,0.616,0.118,0.367,0.357,0,16,60.376
"Timestral 209M + weather, ctx 17520",208.460,137.110,988.640,0.954,0.621,0.120,0.371,0.361,1,17,28.535
"Timestral 209M + weather, ctx 26280",208.880,138.052,987.026,0.952,0.622,0.121,0.372,0.360,0,17,44.112
"Timestral 209M + weather + sun, ctx 32768",207.726,144.346,991.392,0.948,0.619,0.126,0.373,0.362,0,16,97.216
"Timestral 209M + weather + sun, ctx 17520",209.350,144.657,1006.759,0.950,0.624,0.127,0.375,0.367,0,17,46.345
"Timestral 209M + weather + sun, ctx 26280",209.905,144.292,1003.626,0.946,0.625,0.126,0.376,0.366,1,16,73.928
"Timestral 209M + weather, ctx 8760",213.452,149.757,1078.502,0.948,0.636,0.131,0.383,0.394,1,16,12.795
"Timestral 209M + weather + sun, ctx 8760",218.142,157.571,1095.674,0.944,0.650,0.138,0.394,0.400,1,16,21.155
"Timestral 209M + weather, ctx 8760 log",220.212,155.889,1076.452,0.931,0.656,0.137,0.396,0.393,2,16,12.823
"Timestral 209M + weather + sun, ctx 4380 log",218.539,167.409,1144.731,0.950,0.651,0.147,0.399,0.418,0,15,9.791


  freed Timestral 209M: 0.02 GB of GPU memory still in use
[14.7 min since the start of Section 2]

=== Timestral 123M ===
foundation: 8 forecaster(s) x 20 windows


... (15 outputs left out)

  freed TiRex-2: 0.02 GB of GPU memory still in use
[20.2 min since the start of Section 2]

=== TimeGPT ===
foundation: 4 forecaster(s) x 20 windows
  ✓ TimeGPT + weather                              RMSE    220.0  (skill 0.704)   47.9s
  ✓ TimeGPT + weather + sun                        RMSE    211.7  (skill 0.677)   36.3s
  ✓ TimeGPT + weather, ctx 8760                    RMSE    241.9  (skill 0.773)   43.9s
  ✓ TimeGPT + weather + sun, ctx 8760              RMSE    233.0  (skill 0.745)   67.8s


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
TimeGPT + weather,236.054,187.788,1056.583,0.958,0.703,0.165,0.434,0.386,8,14,47.884
TimeGPT + weather + sun,225.022,231.632,1142.625,0.977,0.670,0.203,0.437,0.417,5,15,36.271
"TimeGPT + weather, ctx 8760",263.813,252.863,1132.591,0.962,0.786,0.222,0.504,0.413,0,13,43.879
"TimeGPT + weather + sun, ctx 8760",251.758,312.433,1210.210,0.973,0.750,0.274,0.512,0.442,2,13,67.778
benchmark,335.736,1141.315,2740.292,0.817,1.000,1.000,1.000,1.000,5,0,NaN


  freed TimeGPT: 0.02 GB of GPU memory still in use
[23.6 min since the start of Section 2]


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"Timestral 209M + weather, ctx 32768",206.756,135.137,978.539,0.954,0.616,0.118,0.367,0.357,0,16,60.376
"Timestral 209M + weather, ctx 17520",208.460,137.110,988.640,0.954,0.621,0.120,0.371,0.361,0,17,28.535
"Timestral 209M + weather, ctx 26280",208.880,138.052,987.026,0.952,0.622,0.121,0.372,0.360,0,17,44.112
"Timestral 209M + weather + sun, ctx 32768",207.726,144.346,991.392,0.948,0.619,0.126,0.373,0.362,0,16,97.216
"Timestral 209M + weather + sun, ctx 17520",209.350,144.657,1006.759,0.950,0.624,0.127,0.375,0.367,0,17,46.345
...,...,...,...,...,...,...,...,...,...,...,...
TiRex-2 + weather + sun,250.652,422.025,NaN,NaN,0.747,0.370,0.558,NaN,0,14,1.784
TiRex-2 + weather,250.693,432.003,NaN,NaN,0.747,0.379,0.563,NaN,0,13,2.771
Chronos-2,293.240,575.705,1319.740,0.990,0.873,0.504,0.689,0.482,1,14,2.134
Timestral 209M,295.739,586.581,1237.143,0.965,0.881,0.514,0.697,0.451,0,13,3.430


### 2.2 TabPFN-TS: a pretrained tabular model on the weather and calendar features

TabPFN was pretrained only on synthetic tables — it cannot have seen this series. It gets the same kind of
features as LightGBM on the Mac (hour, day of year, the weather, lags of at least 48 hours) and the last
`TABPFN_DAYS` days as its training table.

In [26]:
TABPFN_DAYS = 120


def tabpfn_ts(transform="log"):
    def fit_predict(x, end, h):
        from tabpfn import TabPFNRegressor
        xs = pd.Series(np.r_[x, np.full(H, np.nan)])

        def design(rows):
            dd = ALL_DATES.iloc[rows]
            cols = {"hour_sin": np.sin(2 * np.pi * dd.dt.hour.to_numpy() / 24), "hour_cos": np.cos(2 * np.pi * dd.dt.hour.to_numpy() / 24),
                    "doy": dd.dt.dayofyear.to_numpy()}
            for k in (48, 72, 168):
                cols[f"lag{k}"] = xs.shift(k).to_numpy()[rows]
            cols["level_7d"] = xs.shift(48).rolling(168).mean().to_numpy()[rows]
            cols.update({c: FEAT_DF[c].to_numpy()[rows] for c in FEAT_COLS})
            return pd.DataFrame(cols)

        rows = np.arange(max(200, end - DAY * TABPFN_DAYS), end)
        model = TabPFNRegressor(device=DEVICE, random_state=1, ignore_pretraining_limits=True)
        model.fit(design(rows), x[rows])
        q = model.predict(design(np.arange(end, end + h)), output_type="quantiles", quantiles=[0.025, 0.5, 0.975])
        return {"mean": q[1], "lo": q[0], "hi": q[2]}
    return on_scale(fit_predict, transform)


run_one_by_one({"TabPFN-TS": {"TabPFN-TS + weather log": tabpfn_ts()}})


=== TabPFN-TS ===
foundation: 1 forecaster(s) x 20 windows
  ✓ TabPFN-TS + weather log                        RMSE    225.1  (skill 0.720)   39.0s


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
TabPFN-TS + weather log,244.854,573.548,1247.710,0.883,0.729,0.503,0.616,0.455,14,14,39.0
benchmark,335.736,1141.315,2740.292,0.817,1.000,1.000,1.000,1.000,6,0,NaN


  freed TabPFN-TS: 0.02 GB of GPU memory still in use
[24.3 min since the start of Section 2]


### 2.3 Neural networks trained on this series (Lab 8, track B)

Unlike Lab 8 (a few hundred monthly points), here there are 33,000 hourly observations and a weather forecast
known for the whole horizon — the setting where neural forecasters earn their keep. `neuralforecast` trains each
network **once on the history before the first window** and then forecasts every window from its own past
(`cross_validation(refit=False)`): no leakage, and one training per model. The weather and the hour of day are
future-known exogenous inputs; `MQLoss` gives a 95% interval of its own. Fitted on the original scale with the weather standardised once (the first
run on the log scale exploded: see the comment in the cell).

In [27]:
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss
from neuralforecast.models import NBEATSx, NHITS, TFT, TiDE

NF_X = COV_COLS + SUN_COLS + ["hour_sin", "hour_cos"]
NF_DAYS = 365                     # <-- days of history the networks learn from (before the first window)
NF_STEPS = 1000                   # <-- training steps per network
SEEDS = [1]                       # <-- add 2, 3 to see how much is the network and how much the seed

# Scales fixed once on the history before the first window (no leakage). Two lessons from the first run:
# - on the log scale an error is multiplied by exp() when transformed back, so one bad window exploded (RMSE x3-x18);
# - per-window scaling of a weather column that hardly moves in the input week turns its horizon values into huge
#   numbers. So: y on its original scale divided by one constant, the weather standardised once, no per-window scaler.
cut = len(y) - ((WINDOWS - 1) * STEP + H)                     # the first window's origin
Y_SCALE = float(np.std(y[:cut]))
nf_frame = pd.DataFrame({"unique_id": "series", "ds": ALL_DATES, "y": np.r_[y, np.full(H, np.nan)] / Y_SCALE})
nf_frame["hour_sin"] = np.sin(2 * np.pi * nf_frame["ds"].dt.hour / 24)
nf_frame["hour_cos"] = np.cos(2 * np.pi * nf_frame["ds"].dt.hour / 24)
for c in COV_COLS + SUN_COLS:
    v = FEAT_DF[c].to_numpy(float)
    nf_frame[c] = (v - v[:cut].mean()) / (v[:cut].std() or 1.0)
history = nf_frame.iloc[max(0, cut - NF_DAYS * DAY):len(y)].reset_index(drop=True)
future = nf_frame.iloc[len(y):].drop(columns="y").reset_index(drop=True)


def networks(seed):
    common = dict(h=H, futr_exog_list=NF_X, loss=MQLoss(level=[95]), max_steps=NF_STEPS, scaler_type="identity",
                  random_seed=seed, windows_batch_size=256, enable_progress_bar=False, enable_model_summary=False,
                  logger=False, accelerator="gpu" if torch.cuda.is_available() else "cpu")   # CPU outside Colab (Apple's MPS runs out of memory)
    return [NHITS(input_size=7 * DAY, alias=f"NHITS s{seed}", **common),       # TFT dropped: the slowest and the worst
            NBEATSx(input_size=7 * DAY, alias=f"NBEATSx s{seed}", **common),    # (RMSE skill 1.01 at 10 windows)
            TiDE(input_size=7 * DAY, alias=f"TiDE s{seed}", **common)]


def add_columns(columns: dict, family: str):
    # put externally computed forecasts (in bt's row order) into bt
    global bt
    frame = bt[list(BASE_COLS)].copy()
    for name, (mean, lo, hi) in columns.items():
        frame[name], frame[f"{name} lo"], frame[f"{name} hi"] = mean, lo, hi
        FAMILY[name] = family
    frame.attrs["seconds"] = {}
    bt = add(bt, frame)


back = lambda v: np.maximum(np.asarray(v, float) * Y_SCALE, 0)   # to the original units; generation is never negative
old = [n for n, f in FAMILY.items() if f == "neural"]              # results of an earlier run of this cell
bt = bt.drop(columns=[c for n in old for c in (n, f"{n} lo", f"{n} hi") if c in bt.columns])
for n in old:
    FAMILY.pop(n, None)
    FINAL.pop(n, None)

for seed in SEEDS:
    start = time.time()
    nf = NeuralForecast(models=networks(seed), freq=FREQ)
    cv = nf.cross_validation(df=history, n_windows=WINDOWS, step_size=STEP, refit=False).sort_values(["cutoff", "ds"])
    assert np.allclose(cv["y"].to_numpy() * Y_SCALE, bt["actual"].to_numpy()), "windows differ from bt"
    names = [m.alias for m in nf.models]
    add_columns({f"{n} + weather": (back(cv[f"{n}-median"]), back(cv[f"{n}-lo-95"]), back(cv[f"{n}-hi-95"]))
                 for n in names}, "neural")
    nf.fit(df=nf_frame.iloc[max(0, len(y) - NF_DAYS * DAY):len(y)])          # the whole recent history, for the submission
    fc = nf.predict(futr_df=future.assign(unique_id="series"))
    for n in names:
        FINAL[f"{n} + weather"] = {"mean": back(fc[f"{n}-median"]), "lo": back(fc[f"{n}-lo-95"]), "hi": back(fc[f"{n}-hi-95"])}
    print(f"seed {seed}: {time.time() - start:.0f}s")
display(window_table([n for n, f in FAMILY.items() if f == "neural"]))
elapsed()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


seed 1: 74s


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
NHITS s1 + weather,267.490,411.986,5404.109,0.273,0.797,0.361,0.579,1.972,4,13,NaN
NBEATSx s1 + weather,313.808,329.594,6541.146,0.340,0.935,0.289,0.612,2.387,4,12,NaN
TiDE s1 + weather,294.783,489.726,1469.219,0.925,0.878,0.429,0.654,0.536,7,10,NaN
benchmark,335.736,1141.315,2740.292,0.817,1.000,1.000,1.000,1.000,5,0,NaN


[25.6 min since the start of Section 2]


### 2.4 The scoreboard here, and the files for the Mac notebook

In [28]:
scoreboard, per_window = window_table(per_window=True)
display(scoreboard)
display(per_window.head(15))
export("colab")                                   # results/colab_bt.csv and results/colab_final.csv, on the runtime
RESULTS = [f"{RESULTS_DIR}/colab_bt.csv", f"{RESULTS_DIR}/colab_final.csv"]
DRIVE_DIR = "/content/drive/MyDrive/MATH840"      # <-- where the results go on your Google Drive

# 1) Google Drive: works in the browser and in VS Code (it asks you to sign in to Google once)
try:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for f in RESULTS:
        shutil.copy(f, DRIVE_DIR)
    print(f"saved {RESULTS} to Google Drive: My Drive/MATH840 -- download them from drive.google.com "
          "and put them in the results/ folder next to LUKIANETS_project_v2.ipynb")
except Exception as err:
    print(f"Google Drive not available ({type(err).__name__}); the files are on the runtime in /content")

# 2) a direct download: works in the browser only (files.download is not supported in VS Code)
try:
    from google.colab import files
    for f in RESULTS:
        files.download(f)
except Exception:
    pass

,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"Timestral 209M + weather, ctx 32768",206.756,135.137,978.539,0.954,0.616,0.118,0.367,0.357,0,16,60.376
"Timestral 209M + weather, ctx 17520",208.460,137.110,988.640,0.954,0.621,0.120,0.371,0.361,0,17,28.535
"Timestral 209M + weather, ctx 26280",208.880,138.052,987.026,0.952,0.622,0.121,0.372,0.360,0,17,44.112
"Timestral 209M + weather + sun, ctx 32768",207.726,144.346,991.392,0.948,0.619,0.126,0.373,0.362,0,16,97.216
"Timestral 209M + weather + sun, ctx 17520",209.350,144.657,1006.759,0.950,0.624,0.127,0.375,0.367,0,17,46.345
...,...,...,...,...,...,...,...,...,...,...,...
TabPFN-TS + weather log,244.854,573.548,1247.710,0.883,0.729,0.503,0.616,0.455,1,14,39.000
TiDE s1 + weather,294.783,489.726,1469.219,0.925,0.878,0.429,0.654,0.536,1,10,NaN
Chronos-2,293.240,575.705,1319.740,0.990,0.873,0.504,0.689,0.482,0,14,2.134
Timestral 209M,295.739,586.581,1237.143,0.965,0.881,0.514,0.697,0.451,0,13,3.430


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
"Timestral 209M + weather, ctx 32768",1.72,0.69,0.49,0.34,1.03,0.76,0.15,0.48,0.24,0.58,0.66,0.48,0.68,0.49,0.71,1.02,1.00,0.43,0.81,1.12
"Timestral 209M + weather, ctx 17520",1.88,0.77,0.58,0.37,1.07,0.80,0.14,0.49,0.22,0.62,0.64,0.42,0.66,0.48,0.65,0.98,0.98,0.44,0.84,1.10
"Timestral 209M + weather, ctx 26280",1.87,0.82,0.55,0.34,1.02,0.77,0.15,0.52,0.21,0.60,0.65,0.45,0.66,0.49,0.68,1.00,0.95,0.43,0.87,1.13
"Timestral 209M + weather + sun, ctx 32768",1.69,0.59,0.46,0.38,1.04,0.74,0.15,0.49,0.25,0.60,0.68,0.52,0.67,0.51,0.77,1.02,0.99,0.42,0.81,1.07
"Timestral 209M + weather + sun, ctx 17520",1.83,0.71,0.58,0.39,1.04,0.76,0.13,0.49,0.23,0.64,0.66,0.45,0.66,0.51,0.72,1.00,0.95,0.38,0.87,1.12
"Timestral 209M + weather + sun, ctx 26280",1.83,0.71,0.54,0.36,1.00,0.75,0.13,0.52,0.22,0.62,0.67,0.48,0.65,0.52,0.74,1.01,0.94,0.41,0.88,1.09
"Timestral 209M + weather, ctx 8760",1.63,0.52,0.30,0.41,1.03,0.76,0.18,0.51,0.19,0.66,0.65,0.42,0.70,0.64,0.91,1.14,1.13,0.44,0.92,0.92
"Timestral 209M + weather + sun, ctx 8760",1.57,0.50,0.34,0.43,0.96,0.70,0.22,0.56,0.20,0.69,0.66,0.47,0.69,0.67,1.06,1.18,1.14,0.42,0.94,0.91
"Timestral 209M + weather, ctx 8760 log",1.70,0.64,0.27,0.58,0.95,0.72,0.18,0.48,0.21,0.71,0.69,0.56,0.79,0.73,1.09,1.08,1.02,0.36,0.93,0.93
"Timestral 123M + weather, ctx 32768",2.16,1.37,0.78,0.39,1.14,0.82,0.27,0.36,0.25,0.61,0.68,0.37,0.60,0.70,0.84,0.98,0.83,0.30,0.78,1.06


saved colab_bt.csv (78 models) and colab_final.csv
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved ['colab_bt.csv', 'colab_final.csv'] to Google Drive: My Drive/MATH840 -- download them from drive.google.com and put them next to LUKIANETS_project_v2.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>